# Time-Series Databases — First Contact

A relational database stores the *current state* of things. A time-series database stores the *history of how things changed over time*. Every row has a timestamp — that is not just a column, it is the primary organizing principle. InfluxDB is built specifically for high-volume time-stamped data: metrics, sensor readings, telemetry. It automatically compresses old data, drops data after a retention period, and answers "what was the average CPU last 5 minutes" without a full table scan.

## What makes time-series databases different

- **Timestamp is first-class** — data is indexed and compressed by time; range queries are the primary access pattern.
- **Tags vs Fields** — tags are indexed metadata (`endpoint_id`, `datacenter`); fields are the actual measurements (`value`). Tags are fast to filter on; high-cardinality tags cause performance problems.
- **Retention policies** — data auto-expires after a defined period; no manual DELETE jobs needed.
- **Downsampling** — summarise old high-resolution data into low-resolution aggregates automatically: 1s → 1min → 1hr → 1day.
- **When to use** — infrastructure metrics, IoT sensors, financial ticks, APM, any data that is always appended and queried by time range.

In [1]:
from pathlib import Path
import sys

for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_influxdb_client
import pandas as pd
import os

client = get_influxdb_client()
health = client.health()
print(f"InfluxDB status:  {health.status}")
print(f"InfluxDB version: {health.version}")

query_api = client.query_api()
org = os.getenv('INFLUXDB_ORG', 'de_org')
print("Query API ready.")

InfluxDB status:  pass
InfluxDB version: v2.7.12
Query API ready.


In [2]:
# Explore what is in InfluxDB
# InfluxDB data model: measurement → tags (indexed) → fields (values)
# Our seed: measurement='metrics', tags={endpoint_id, metric_name, unit}, field='value'

buckets_api = client.buckets_api()
buckets = buckets_api.find_buckets().buckets
print("Buckets:")
for b in buckets:
    if not b.name.startswith('_'):
        print(f"  {b.name}")

# Count total points and show the data model via a sample record
sample_q = '''
from(bucket: "telemetry")
  |> range(start: -91d)
  |> filter(fn: (r) => r._measurement == "metrics")
  |> limit(n: 1)
'''
tables = query_api.query(sample_q, org=org)
for table in tables:
    for record in table.records:
        print("\nSample record:")
        print(f"  measurement : {record.get_measurement()}")
        print(f"  time        : {record.get_time()}")
        print(f"  field       : {record.get_field()} = {record.get_value()}")
        tags = {k: v for k, v in record.values.items()
                if not k.startswith('_') and k not in ('result', 'table')}
        print(f"  tags        : {tags}")
    break

# Total point count
count_q = '''
from(bucket: "telemetry")
  |> range(start: -91d)
  |> filter(fn: (r) => r._measurement == "metrics")
  |> count()
'''
tables = query_api.query(count_q, org=org)
total = sum(r.get_value() for t in tables for r in t.records)
print(f"\nTotal metric points in InfluxDB: {total:,}")

Buckets:
  telemetry

Sample record:
  measurement : metrics
  time        : 2026-03-09 17:28:17.063609+00:00
  field       : value = 955.02
  tags        : {'endpoint_id': '002563b4-312a-4577-83fa-ed9a11f75fae', 'metric_name': 'network_out', 'unit': 'mbps'}

Total metric points in InfluxDB: 5,000


## 5 Flux queries against the telemetry time-series

Flux reads like a pipeline: `from(bucket) |> range(start, stop) |> filter() |> aggregateWindow() |> yield()`  
Every query **must** start with `from()` and `range()` — InfluxDB refuses unbounded scans by design.

**Flux data model note:** `metric_name` is a *tag* (indexed string), `value` is the *field* (the numeric measurement). In Flux, tags are accessed as `r.tag_name` and the field value is `r._value`.

In [3]:
# Query 1 — sample raw metric points (2 per series)
# Note: Flux sort()+limit() operates per-table (per series), not globally.
# limit(n:2) here gives 2 points from each metric_name series.
q1 = '''
from(bucket: "telemetry")
  |> range(start: -91d)
  |> filter(fn: (r) => r._measurement == "metrics")
  |> limit(n: 2)
'''
tables = query_api.query(q1, org=org)
rows = []
for table in tables:
    for record in table.records:
        rows.append({
            'time':        str(record.get_time())[:19],
            'metric_name': record.values.get('metric_name', ''),
            'value':       round(record.get_value(), 2),
            'unit':        record.values.get('unit', ''),
            'endpoint_id': str(record.values.get('endpoint_id', ''))[:8] + '...'
        })
df1 = pd.DataFrame(rows)
print(f"Sample raw points ({len(df1)} rows, 2 per metric series):")
print(df1.to_string(index=False))

Sample raw points (4990 rows, 2 per metric series):
               time    metric_name   value    unit endpoint_id
2026-03-09 17:28:17    network_out  955.02    mbps 002563b4...
2026-03-22 15:16:26    network_out  165.52    mbps 00568a4e...
2026-03-02 01:47:46     network_in  604.59    mbps 0063e0f2...
2025-12-24 06:45:38    cpu_percent   83.71 percent 006e59bf...
2026-03-11 19:23:01     network_in  218.86    mbps 006e59bf...
2026-03-22 21:37:44     network_in  995.55    mbps 00712f16...
2026-03-05 23:49:15        disk_io 2120.09    iops 0074741f...
2026-02-16 08:41:57     network_in  292.77    mbps 0093b002...
2026-02-16 23:56:12        disk_io  731.19    iops 00a4e88c...
2026-02-09 10:32:52        disk_io 1477.97    iops 00a5e840...
2025-12-23 19:00:04    cpu_percent   98.38 percent 00c60d6b...
2026-01-20 04:38:35    cpu_percent    9.64 percent 00c60d6b...
2026-03-22 08:05:09 memory_percent   55.84 percent 00d011d2...
2026-02-02 22:25:32        disk_io  975.37    iops 00e249e4...
202

In [4]:
# Query 2 — average value by metric_name over last 90 days
# group() regroups data by tag; mean() computes mean per group
q2 = '''
from(bucket: "telemetry")
  |> range(start: -91d)
  |> filter(fn: (r) => r._measurement == "metrics")
  |> group(columns: ["metric_name"])
  |> mean()
'''
tables = query_api.query(q2, org=org)
rows = []
for table in tables:
    for record in table.records:
        rows.append({
            'metric_name': record.values.get('metric_name', ''),
            'avg_value':   round(record.get_value(), 2)
        })
df2 = pd.DataFrame(rows).sort_values('avg_value', ascending=False)
print("Average value by metric_name (last 90 days):")
print(df2.to_string(index=False))

Average value by metric_name (last 90 days):
   metric_name  avg_value
       disk_io    2529.33
    network_in     500.22
   network_out     496.20
memory_percent      53.96
   cpu_percent      51.19


In [5]:
# Query 3 — downsample CPU metrics into daily averages
# aggregateWindow is the signature time-series operation —
# collapses raw points into lower-resolution summaries.
# SQL equivalent: GROUP BY DATE_TRUNC('day', recorded_at) — but built-in here.
q3 = '''
from(bucket: "telemetry")
  |> range(start: -91d)
  |> filter(fn: (r) => r._measurement == "metrics"
                    and r.metric_name == "cpu_percent")
  |> aggregateWindow(every: 7d, fn: mean, createEmpty: false)
  |> sort(columns: ["_time"])
'''
tables = query_api.query(q3, org=org)
rows = []
for table in tables:
    for record in table.records:
        rows.append({
            'week_start': str(record.get_time())[:10],
            'avg_cpu':    round(record.get_value(), 2)
        })
df3 = pd.DataFrame(rows).drop_duplicates('week_start').sort_values('week_start')
print("Weekly average CPU (last 90 days, downsampled from raw points):")
print(df3.to_string(index=False))

Weekly average CPU (last 90 days, downsampled from raw points):
week_start  avg_cpu
2025-12-25    83.71
2026-01-01    60.18
2026-01-08    96.21
2026-01-15    39.15
2026-01-22     9.64
2026-01-29     9.97
2026-02-05    86.36
2026-02-12    27.80
2026-02-19    60.37
2026-02-26    98.45
2026-03-05    31.22
2026-03-12    72.16
2026-03-19    27.37
2026-03-24    75.38


In [6]:
# Query 4 — find metric spikes (CPU > 80%)
# filter() on _value is the Flux equivalent of WHERE value > 80
q4 = '''
from(bucket: "telemetry")
  |> range(start: -91d)
  |> filter(fn: (r) => r._measurement == "metrics"
                    and r.metric_name == "cpu_percent"
                    and r._value > 80.0)
  |> sort(columns: ["_time"], desc: true)
  |> limit(n: 10)
'''
tables = query_api.query(q4, org=org)
rows = []
for table in tables:
    for record in table.records:
        rows.append({
            'time':        str(record.get_time())[:19],
            'endpoint_id': str(record.values.get('endpoint_id', ''))[:8] + '...',
            'cpu_value':   round(record.get_value(), 1)
        })
df4 = pd.DataFrame(rows)
print(f"High CPU spikes (> 80%) — up to 10 per series:")
if df4.empty:
    print("  (no spikes found in seeded data — try lowering threshold)")
else:
    print(df4.to_string(index=False))

High CPU spikes (> 80%) — up to 10 per series:
               time endpoint_id  cpu_value
2025-12-24 06:45:38 006e59bf...       83.7
2025-12-23 19:00:04 00c60d6b...       98.4
2026-01-29 02:45:34 00e2b519...       86.4
2026-01-01 03:34:20 0171bf76...       96.2
2026-01-19 04:30:19 01bc47b1...       88.9
2026-02-22 22:20:20 08f3c75e...       98.5
2026-03-06 17:10:00 0cbeb48a...       84.3
2026-01-30 17:32:53 0ccbace9...       85.7
2026-01-18 01:36:01 0d1ab160...       86.9
2026-02-24 04:28:13 0d79c52d...       92.1
2026-02-12 19:50:43 0d89bcdd...       80.8
2026-02-03 23:28:41 0dec09d3...       86.2
2026-02-26 22:51:02 0f8d6d4b...       80.5
2026-03-21 00:46:47 10a88e5e...       82.8
2025-12-28 10:51:36 11d2a7a6...       89.3
2026-01-09 11:46:57 14eaaeb1...       80.7
2026-03-09 16:51:31 15faf379...       94.2
2026-03-09 09:13:33 19c49417...       81.5
2026-03-12 09:44:07 1a0a95ad...       80.3
2026-01-13 17:57:30 1b884ceb...       85.0
2026-01-24 08:32:51 1bacebd1...       89.3
2025-12

In [7]:
# Query 5 — InfluxDB vs Postgres timing comparison
# Count cpu_percent points over the full range in both engines.
# Note: InfluxDB was seeded with 5K demo points; Postgres has the full 500K rows.
# The comparison shows query mechanics, not apples-to-apples volume.
import time
from db_connections import get_postgres_conn

# InfluxDB
t0 = time.perf_counter()
tables = query_api.query('''
    from(bucket: "telemetry")
      |> range(start: -91d)
      |> filter(fn: (r) => r._measurement == "metrics"
                        and r.metric_name == "cpu_percent")
      |> count()
''', org=org)
influx_count = sum(r.get_value() for t in tables for r in t.records)
influx_ms = (time.perf_counter() - t0) * 1000

# Postgres
pg = get_postgres_conn()
cur = pg.cursor()
t0 = time.perf_counter()
cur.execute("""
    SELECT COUNT(*) FROM telemetry.metrics
    WHERE metric_name = 'cpu_percent'
    AND recorded_at >= NOW() - INTERVAL '91 days'
""")
pg_count = cur.fetchone()[0]
pg_ms = (time.perf_counter() - t0) * 1000

print("Count of cpu_percent points (last 91 days):")
print(f"  InfluxDB   : {influx_count:>6,} points  in {influx_ms:>6.0f}ms  (demo seed — 5K total points)")
print(f"  PostgreSQL : {pg_count:>6,} rows    in {pg_ms:>6.0f}ms  (full seed — 500K total rows)")
print()
print("At production scale (billions of points), InfluxDB's time-indexed")
print("columnar storage and automatic downsampling outperform a relational table.")

Count of cpu_percent points (last 91 days):
  InfluxDB   :    983 points  in    119ms  (demo seed — 5K total points)
  PostgreSQL : 100,013 rows    in     31ms  (full seed — 500K total rows)

At production scale (billions of points), InfluxDB's time-indexed
columnar storage and automatic downsampling outperform a relational table.


## SQL vs Flux — same question, different language

| SQL (Postgres) | Flux (InfluxDB) |
|---|---|
| `WHERE recorded_at > NOW()-'1h'` | `\|> range(start: -1h)` |
| `WHERE metric_name = 'cpu_percent'` | `\|> filter(fn: (r) => r.metric_name == "cpu_percent")` |
| `AVG(value) GROUP BY DATE_TRUNC('day', recorded_at)` | `\|> aggregateWindow(every: 1d, fn: mean)` |
| `WHERE value > 80` | `\|> filter(fn: (r) => r._value > 80.0)` |
| `ORDER BY recorded_at DESC LIMIT 10` | `\|> sort(columns: ["_time"], desc: true) \|> limit(n: 10)` |
| Manual DELETE or partition drop | Retention policy — auto-expire after N days |

## Key observations

- **Flux pipeline model** — every query is a chain of pipe operators; `from()` → `range()` → `filter()` → aggregate → `yield()`. Nothing runs without a time range — no unbounded scans by design.
- **Tags are indexed, fields are not** — tag high-cardinality strings (`endpoint_id`, `metric_name`) as tags so filters are fast. Never use a UUID-generating value as a tag with millions of distinct values — that's *cardinality explosion*, the most common InfluxDB performance killer.
- **`aggregateWindow` is the killer feature** — one operator does what Postgres needs `GROUP BY DATE_TRUNC + subquery` to achieve, and it's built into the storage engine's compression model.
- **`sort() |> limit()` is per-series** — unlike SQL `ORDER BY ... LIMIT`, Flux operates per table (series). To get a true global top-N you need `group()` first to merge tables, then sort. This is a common Flux gotcha.
- **Citi hook** — 6,000 endpoints each emitting 5 metrics every 10 seconds = 3,000 writes/second = 260M rows/day. Postgres starts groaning at that volume. InfluxDB handles it trivially: time-series compression, automatic downsampling tasks (raw → 1min → 1hr → 1day), and retention policies mean you never manually manage data lifecycle.